# 🚀 DataMate — Zalo AI Challenge 2026: SFT & GRPO Training Pipeline

Notebook này hướng dẫn quy trình huấn luyện nâng cao cho mô hình **Qwen2.5-Coder-7B-Instruct** dựa trên tập dữ liệu `train` (30 doanh nghiệp, 3,000 câu hỏi có sẵn đáp án `answers.jsonl`):

1. **Pre-build Cache:** Tạo cơ sở dữ liệu DuckDB tốc độ cao cho toàn bộ tập `train`.
2. **Rejection Sampling (RFT):** Tự động sinh code (kèm cơ chế tự sửa lỗi 2 lượt) và lọc các mẫu đúng 100% để tạo tập dữ liệu vàng (`train/sft_dataset.jsonl`).
3. **Supervised Fine-Tuning (SFT LoRA):** Huấn luyện mô hình nắm vững cú pháp DuckDB tiếng Việt (Completion-only Loss Masking).
4. **GRPO Reinforcement Learning:** Huấn luyện học tăng cường tự sửa sai với hàm thưởng khách quan (Rule-based Verifiable Outcome Reward).
5. **Suy luận & Nộp bài:** Chạy mô hình đã fine-tune trên `public_test` để xuất file nộp bài chuẩn.

## 1. Kiểm tra thông tin GPU & Cài đặt môi trường

In [11]:
# 1.1 Kiểm tra GPU (Khuyến nghị dùng A100 GPU trên Colab Pro)
!nvidia-smi

Sun Nov  1 17:25:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             49W /  400W |     662MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 1.2 Cài đặt các thư viện cần thiết
!pip install -q -U transformers accelerate datasets peft 'trl>=0.14.0' bitsandbytes duckdb unidecode tqdm python-calamine

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 2. Kết nối Google Drive & Đồng bộ mã nguồn (Branch dev)

In [3]:
# 2.1 Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ZALO_AI_Challenge'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

Mounted at /content/drive
/content/drive/MyDrive/ZALO_AI_Challenge


In [4]:
# 2.2 Đồng bộ code mới nhất từ GitHub nhánh dev
import os
if not os.path.exists('.git'):
    !git init
    !git remote add origin https://github.com/JimmyK300/zalo-vecna.git

!git fetch origin dev && git reset --hard origin/dev
!git status

remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 14 (delta 10), reused 12 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 3.10 KiB | 0 bytes/s, done.
From https://github.com/JimmyK300/zalo-vecna
 * branch            dev        -> FETCH_HEAD
   aa21e35..f336217  dev        -> origin/dev
Updating files: 100% (16/16), done.
HEAD is now at f336217 Update Cell 8 in DataMate_Train_GRPO_Colab.ipynb for <4h completion
On branch dev
Your branch is up to date with 'origin/dev'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	=0.43.0
	=1.0.0
	=1.1.0
	=2.2.0
	=4.48.0
	submission_sample.jsonl

nothing added to commit but untracked files present (use "git add" to track)


## 3. Tiền xử lý DuckDB Cache cho Tập `train`

In [5]:
# ⚡ 3.1 Build cache DuckDB cho tập train (nếu đã chạy trước đó thì KHÔNG CẦN chạy lại, code mới sẽ tự động sửa cache trong 0.01s!)
!python prebuild_cache.py --data_dir train

[*] Found 30 businesses in train/data.
[*] Pre-building high-speed DuckDB disk cache (.duckdb)...


[+] All 30 businesses cached in 43.39 seconds!
[+] Subsequent pipeline runs will load in ~0.01s per business!


## 4. Rejection Sampling (RFT) — Khai thác dữ liệu vàng (3,000 câu)

In [ ]:
%cd /content/drive/MyDrive/ZALO_AI_Challenge
!git fetch origin dev && git reset --hard origin/dev

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python build_training_dataset.py \
    --data_dir train \
    --output train/sft_dataset.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --samples_per_question 1 \
    --max_golden 180 \
    --batch_size 16 \
    --fp16 && \
python train_sft.py \
    --data_path train/sft_dataset.jsonl \
    --output_dir checkpoints/sft_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --max_seq_length 4096 \
    --lr 2e-4 \
    --fp16 && \
python run_public_test.py \
    --data_dir public_test \
    --output submission_sft.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/sft_adapter \
    --batch_size 16 \
    --fp16 && \
python train_grpo.py \
    --data_dir train \
    --adapter_path checkpoints/sft_adapter \
    --sft_data_path train/sft_dataset.jsonl \
    --output_dir checkpoints/grpo_adapter \
    --limit 350 \
    --num_generations 4 \
    --epochs 1 \
    --lr 5e-6 \
    --fp16 && \
python run_public_test.py \
    --data_dir public_test \
    --output submission_grpo.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/grpo_adapter \
    --batch_size 16 \
    --fp16

## 5. Supervised Fine-Tuning (SFT với LoRA)

In [ ]:
# 🧠 5.1 Huấn luyện LoRA SFT trên tập dữ liệu vàng (mất ~20-30 phút trên A100)
!python train_sft.py \
    --data_path train/sft_dataset.jsonl \
    --output_dir checkpoints/sft_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --max_seq_length 4096 \
    --lr 2e-4 \
    --bf16

## 6. Group Relative Policy Optimization (GRPO)

In [ ]:
# 🎯 6.1 Huấn luyện học tăng cường GRPO với DuckDB Verifiable Reward (Warm-start từ SFT)
!python train_grpo.py \
    --data_dir train \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/sft_adapter \
    --output_dir checkpoints/grpo_adapter \
    --num_generations 4 \
    --epochs 1 \
    --batch_size 4 \
    --grad_accum 2 \
    --lr 5e-6 \
    --bf16 \
    --limit 500

## 7. Đánh giá & Suy luận trên `public_test` với Model đã Fine-tune

In [ ]:
# 🚀 7.1 Chạy suy luận Public Test với LoRA Adapter đã huấn luyện (ưu tiên grpo_adapter)
!python run_public_test.py \
    --data_dir public_test \
    --output submission_finetuned.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/grpo_adapter \
    --batch_size 16 \
    --fp16 \
    --overwrite

# Xem thử 10 câu đầu tiên
!head -n 10 submission_finetuned.jsonl

## 8. 🌙 Chế độ Treo Máy Qua Đêm (Chạy liên hoàn RFT -> SFT -> GRPO -> Public Test)

In [ ]:
%cd /content/drive/MyDrive/ZALO_AI_Challenge
!git fetch origin dev && git reset --hard origin/dev

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python build_training_dataset.py \
    --data_dir train \
    --output train/sft_dataset.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --samples_per_question 1 \
    --max_golden 180 \
    --batch_size 16 \
    --fp16 && \
python train_sft.py \
    --data_path train/sft_dataset.jsonl \
    --output_dir checkpoints/sft_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --max_seq_length 4096 \
    --lr 2e-4 \
    --fp16 && \
python run_public_test.py \
    --data_dir public_test \
    --output submission_sft.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/sft_adapter \
    --batch_size 16 \
    --fp16 && \
python train_grpo.py \
    --data_dir train \
    --adapter_path checkpoints/sft_adapter \
    --sft_data_path train/sft_dataset.jsonl \
    --output_dir checkpoints/grpo_adapter \
    --limit 350 \
    --num_generations 4 \
    --epochs 1 \
    --lr 5e-6 \
    --fp16 && \
python run_public_test.py \
    --data_dir public_test \
    --output submission_grpo.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/grpo_adapter \
    --batch_size 16 \
    --fp16

/content/drive/MyDrive/ZALO_AI_Challenge
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 707 bytes | 1024 bytes/s, done.
From https://github.com/JimmyK300/zalo-vecna
 * branch            dev        -> FETCH_HEAD
   f7a8ac2..ea52e2b  dev        -> origin/dev
HEAD is now at ea52e2b Support seamless resume from checkpoint-100 in train_grpo.py
[*] Found 181 existing golden samples in train/sft_dataset.jsonl (fast-forwarding to index 2164/3000).
[+] Already have 181 >= 180 golden samples. Skipping mining!
[+] Found existing trained LoRA adapter at: checkpoints/sft_adapter. Skipping SFT training!
=== Starting DataMate Inference ===
[*] Using data directory: /content/drive/MyDrive/ZALO_AI_Challenge/public_test
[*] Initializing model: Qwen/Qwen2.5-Coder-7B-Instruct on cuda (4-bit: False)
[transformers] `torch_dtype

: 

: 

In [12]:
%cd /content/drive/MyDrive/ZALO_AI_Challenge
!git fetch origin dev && git reset --hard origin/dev

!python run_public_test.py \
    --data_dir public_test \
    --output submission_grpo.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/grpo_adapter \
    --batch_size 16 \
    --fp16 \
    --overwrite

/content/drive/MyDrive/ZALO_AI_Challenge
From https://github.com/JimmyK300/zalo-vecna
 * branch            dev        -> FETCH_HEAD
HEAD is now at ea52e2b Support seamless resume from checkpoint-100 in train_grpo.py
=== Starting DataMate Inference ===
[*] Using data directory: /content/drive/MyDrive/ZALO_AI_Challenge/public_test
[*] Initializing model: Qwen/Qwen2.5-Coder-7B-Instruct on cuda (4-bit: False)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

[*] Loading fine-tuned LoRA adapter from: checkpoints/grpo_adapter
[+] Model loaded successfully!
[*] Processing all 1000 questions.

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py", line 3000, in generate
    result = decoding_method(
        self,
    ...<5 lines>...
        **model_kwargs,
    )
  File "/usr/loc